In [0]:
from lib.api_extraction import downlaod_stock
from datetime import datetime, timedelta
from pyspark.sql.functions import lit
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, TimestampType, LongType
from datetime import datetime

In [0]:
# wigdet text, gdzie mozesz wpisac nazwe akcji
dbutils.widgets.text("ticker", "")
# get value from widget
ticker = dbutils.widgets.get("ticker")
assert ticker != "", "Ticker is empty"

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("job_id", "")

run_id = dbutils.widgets.get("run_id")
job_id = dbutils.widgets.get("job_id")

In [0]:
end_date = datetime.today()
start_date = end_date - timedelta(days=40*365)




In [0]:
%sql
CREATE table if not exists databricksformula1.default.table_runs (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    ingestion_date VARCHAR(255),
    created_timestamp TIMESTAMP,
    run_id INT,
    job_id INT,
    ticker VARCHAR(255),
    extraction_from DATE,
    extraction_to DATE,
    status VARCHAR(255)
    
);


In [0]:
stocks_schema = StructType([
    StructField("ingestion_date", StringType(), True),
    StructField("created_timestamp", TimestampType(), True),
    StructField("run_id", LongType(), True),
    StructField("job_id", LongType(), True),
    StructField("ticker", StringType(), True),
    StructField("extraction_from", DateType(), True),
    StructField("extraction_to", DateType(), True),
    StructField("status", StringType(), True)
    
])


In [0]:
def prepare_df_with_stock(ticker, start_date, end_date):
    pdf = downlaod_stock(ticker, start_date, end_date)
    pdf = pdf.reset_index() #pdf.reset_index(inplace=True)
    new_columns = ['Date', 'Close', 'High', 'Low', 'Open', 'Volume']
    df = spark.createDataFrame(pdf, new_columns)
# cast into int run_id in pyspark

    df = df.withColumn("job_id", lit(job_id).cast("long"))
    df = df.withColumn("run_id", lit(run_id).cast("long"))

    df = df.withColumn("ticker", lit(ticker))
    df = df.withColumn("created_timestamp", lit(datetime.now()))
    df = df.withColumn("ingestion_date", lit(datetime.now().strftime("%Y-%m-%d")))

    return df

try:
    df = prepare_df_with_stock(ticker, start_date, end_date)
    df.write.mode("append").option("mergeSchema", "true").saveAsTable("databricksformula1.default.stocks")
    metadata_df = spark.createDataFrame([(datetime.now().strftime("%Y-%m-%d"), datetime.now(), int(run_id), int(job_id), ticker, start_date, end_date, "Success")], stocks_schema)
    metadata_df.write.mode("append").saveAsTable("databricksformula1.default.stocks_metadata")
except Exception as e:
    print(e)
    metadata_df = spark.createDataFrame([(datetime.now().strftime("%Y-%m-%d"), datetime.now(), int(run_id), int(job_id), ticker, start_date, end_date, "Fail")], stocks_schema)
    metadata_df.write.mode("append").saveAsTable("databricksformula1.default.stocks_metadata")